### Data Quality Module — Usage Examples
The `quality` module provides reusable tools to validate, profile, and assess the health of your datasets before they move into Delta Lake or downstream analytics.

These helpers are designed for real insurance pipelines, where data quality is critical for compliance, pricing, underwriting, and fraud detection.

### Import Libraries
-----

In [ ]:
from pyspark.sql import SparkSession
from pyspark_utils.cleaning import (
    column_names,
    nulls,
    deduplication,
    dates,
    type_casting,
    text_cleaning
)

### Example Dataset
---

In [ ]:
data = [
    ("POL123", 1000.0, "ACTIVE"),
    ("POL123", None, "ACTIVE"),
    ("POL456", 2000.0, "PENDING"),
    ("POL789", -50.0, "CLOSED"),
]
columns = ["policy_number", "premium", "status"]

df = spark.createDataFrame(data, columns)


### Validators
---

**Check for nulls**

In [ ]:
from pyspark_utils.quality import validators

null_report = validators.check_nulls(df, ["policy_number", "premium"])
null_report.show()


**Check uniqueness**

In [ ]:
is_unique = validators.check_unique(df, ["policy_number"])
print("Unique policy numbers:", is_unique)


**Check numeric ranges**

In [ ]:
premium_ok = validators.check_value_ranges(df, "premium", 0, 100000)
print("Premium values valid:", premium_ok)


### Expectations
---

**Expect non‑null values**

In [ ]:
from pyspark_utils.quality import expectations

premium_not_null = expectations.expect_non_null(df, "premium")


**Expect positive numeric values**

In [ ]:
from pyspark_utils.quality import expectations

premium_not_null = expectations.expect_non_null(df, "premium")


**Expect valid status values**

In [ ]:
status_ok = expectations.expect_valid_status(
    df,
    column="status",
    allowed=["ACTIVE", "PENDING", "CLOSED"]
)


### Profiling
---

**Profile the entire DataFrame**

In [ ]:
from pyspark_utils.quality import profiling

profile = profiling.profile_dataframe(df)
print(profile)


This returns a dictionary with:
- data type
- distinct count
- min/max values
- Perfect for exploratory analysis or automated checks.

### Reports
---

**Generate a quality report**

In [ ]:
from pyspark_utils.quality import reports

results = {
    "Null Check": premium_not_null,
    "Premium Positive": premium_positive,
    "Valid Status": status_ok,
    "Premium Range": premium_ok
}

print(reports.generate_quality_report(results))


Example Output

Data Quality Report:

----------------------------------------  
Null Check: FAIL  
Premium Positive: FAIL  
Valid Status: PASS  
Premium Range: FAIL  
